<a href="https://colab.research.google.com/github/2catch2/ToroidalStateEngine/blob/main/Vortexus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ========================================================
# MICROCODE STATE ENGINE SIMULATION WITH OVERWRITE LOOPS
# ========================================================

def run_microcode_matrix_simulation():
    # 8-slot memory ring (Layer 2 / The Past) starting fully neutral
    past_ring = ["00"] * 8
    ring_index = 0

    # 16 clock cycles of incoming data (Layer 3 / The Future)
    future_stream = [
        "01","10","01","01","10","00","01","10",  # Cycles 0-7: Initial Fill
        "10","10","00","10","01","01","00","01"   # Cycles 8-15: The Overwrite Wave
    ]

    # Microcode opcodes assigned per cycle to test different hardware policies
    microcode_opcodes = [
        "00","00","00","00","00","00","00","00",  # Cycles 0-7: Passive Collapse
        "01","11","01","00","01","11","00","01"   # Cycles 8-15: Dynamic Overwrite & Protect Mix
    ]

    print("=== STARTING MULTI-CYCLE OVERWRITE SIMULATION ===")

    for cycle in range(16):
        incoming = future_stream[cycle]
        current_past = past_ring[ring_index]
        opcode = microcode_opcodes[cycle]

        # --- THE MICROCODE STATE ENGINE MATRIX LOGIC ---
        if opcode == "00":    # PASSIVE COLLAPSE
            if current_past == "00":
                next_state = incoming
            else:
                next_state = current_past # Protect past if occupied

        elif opcode == "01":  # FORCE OVERWRITE
            next_state = incoming         # Future erases past

        elif opcode == "11":  # PROTECT HISTORY
            next_state = current_past     # Lock current state completely

        else:
            next_state = current_past

        # Write to the physical ring loop
        action_taken = "OVERWRITE" if next_state != current_past and current_past != "00" else "WRITE/HOLD"
        past_ring[ring_index] = next_state

        # Display tracking metrics every 4 cycles to save console space, or run full trace
        print(f"Cycle {cycle:02d} | Opcode: {opcode} | In: {incoming} | Past: {current_past} -> Next: {next_state} [{action_taken}]")
        if (cycle + 1) % 8 == 0 or cycle == 15:
            print(f"--> End of Loop Wave Ring State: {past_ring}\n")

        # Move pointer and wrap around perfectly using toroidal modulo geometry
        ring_index = (ring_index + 1) % len(past_ring)

    print("=== SIMULATION SEQUENCE COMPLETE ===")

run_microcode_matrix_simulation()

=== STARTING MULTI-CYCLE OVERWRITE SIMULATION ===
Cycle 00 | Opcode: 00 | In: 01 | Past: 00 -> Next: 01 [WRITE/HOLD]
Cycle 01 | Opcode: 00 | In: 10 | Past: 00 -> Next: 10 [WRITE/HOLD]
Cycle 02 | Opcode: 00 | In: 01 | Past: 00 -> Next: 01 [WRITE/HOLD]
Cycle 03 | Opcode: 00 | In: 01 | Past: 00 -> Next: 01 [WRITE/HOLD]
Cycle 04 | Opcode: 00 | In: 10 | Past: 00 -> Next: 10 [WRITE/HOLD]
Cycle 05 | Opcode: 00 | In: 00 | Past: 00 -> Next: 00 [WRITE/HOLD]
Cycle 06 | Opcode: 00 | In: 01 | Past: 00 -> Next: 01 [WRITE/HOLD]
Cycle 07 | Opcode: 00 | In: 10 | Past: 00 -> Next: 10 [WRITE/HOLD]
--> End of Loop Wave Ring State: ['01', '10', '01', '01', '10', '00', '01', '10']

Cycle 08 | Opcode: 01 | In: 10 | Past: 01 -> Next: 10 [OVERWRITE]
Cycle 09 | Opcode: 11 | In: 10 | Past: 10 -> Next: 10 [WRITE/HOLD]
Cycle 10 | Opcode: 01 | In: 00 | Past: 01 -> Next: 00 [OVERWRITE]
Cycle 11 | Opcode: 00 | In: 10 | Past: 01 -> Next: 01 [WRITE/HOLD]
Cycle 12 | Opcode: 01 | In: 01 | Past: 10 -> Next: 01 [OVERWRITE]

In [ ]:

# =====================================================================
# UNIFIED MASTER SYSTEM TEMPLATE: GLOBAL RESET & CONCENTRIC POWER ADAPT
# =====================================================================
# Sandbox Run: SUCCESSFUL (Zero-Contention Multi-Cycle Trace)
# =====================================================================

class MasterTorusMemoryBlock:
    def __init__(self, size=8):
        self.size = size
        # Initialized cleanly via Concentric Power Grid Start
        self.past_ring = ["00"] * size
        self.ring_index = 0
        self.angle_map = {"000":0,"001":1,"010":2,"011":3,"100":4,"101":5,"110":6,"111":7}

    def execute_clock_cycle(self, incoming_data, microcode_opcode, stack_en=1, cpu_address="000", read_en=0, reset_n=1):
        """
        Executes one hardware clock cycle with global reset overrides.
        reset_n = 0 : Global Flash-Clear active (Instant systemic reset to '00')
        reset_n = 1 : Normal production operation
        """
        # --- LAYER 0: GLOBAL RESET OVERRIDE GATES ---
        if reset_n == 0:
            self.past_ring = ["00"] * self.size # Instant 1-cycle global discharge
            self.ring_index = 0
            return "Hi-Z", "⚡ [GLOBAL RESET_N ACTIVE] All slots flushed to '00' (Neutral Potentials)"

        # --- WRITE PATH: ADDRESS SELECTION DECODER ---
        target_slot = self.angle_map.get(cpu_address, 0) if stack_en == 0 else self.ring_index
        current_past = self.past_ring[target_slot]

        # --- WRITE PATH: MICROCODE & SELF-HEALING ERROR LOGIC ---
        if incoming_data == "11" or current_past == "11":
            neighbor_index = (target_slot - 1) % self.size
            next_state = self.past_ring[neighbor_index]
        else:
            if microcode_opcode == "00":
                next_state = incoming_data if current_past == "00" else current_past
            elif microcode_opcode == "01":
                next_state = incoming_data
            elif microcode_opcode == "11":
                next_state = current_past
            else:
                next_state = current_past

        self.past_ring[target_slot] = next_state

        # --- READ PATH: TRI-STATE SYSTEM BUS OUT ---
        if read_en == 1:
            read_slot = self.angle_map.get(cpu_address, 0) if stack_en == 0 else target_slot
            output_bus = self.past_ring[read_slot]
            telemetry = f"📋 READ Slot {read_slot} Value: {output_bus}"
        else:
            output_bus = "Hi-Z"
            telemetry = "💤 READ BUS DISABLED (High-Impedance Mode)"

        if stack_en == 1:
            self.ring_index = (self.ring_index + 1) % self.size

        return output_bus, telemetry


# =====================================================================
# SYSTEM COMPREHENSIVE EXECUTION TRACE
# =====================================================================
def run_master_test_bench():
    engine = MasterTorusMemoryBlock()

    # 8-cycle simulation trace testing production workloads and global clear
    execution_profile = [
        # In,   Opcode, STACK_EN, CPU_ADDR, READ_EN, RESET_N
        ("01",   "01",     1,      "000",     1,       1),   # Cycle 0: Fill slot 0 and read it
        ("10",   "01",     1,      "000",     1,       1),   # Cycle 1: Fill slot 1 and read it
        ("01",   "01",     1,      "000",     1,       1),   # Cycle 2: Fill slot 2 and read it
        ("00",   "11",     1,      "000",     0,       0),   # Cycle 3: TRIGGER GLOBAL RESET!
        ("01",   "00",     1,      "000",     1,       1),   # Cycle 4: Post-reset auto-fill restarts
        ("10",   "01",     1,      "000",     1,       1),   # Cycle 5: Normal stream continuation
    ]

    print("=== EXECUTING UNIFIED MASTER SIMULATION PROFILE ===\n")
    for cycle, (incoming, opcode, stack_en, cpu_addr, read_en, reset_n) in enumerate(execution_profile):
        bus_out, log = engine.execute_clock_cycle(incoming, opcode, stack_en, cpu_addr, read_en, reset_n)
        print(f"Cycle {cycle:02d} | Hardware Pin Output: [{bus_out}]")
        print(f"         ├─ Tracker: {log}")
        print(f"         └─ Memory Ring Grid Array: {engine.past_ring}\n")

    print("=====================================================")

run_master_test_bench()

=== EXECUTING UNIFIED MASTER SIMULATION PROFILE ===

Cycle 00 | Hardware Pin Output: [01]
         ├─ Tracker: 📋 READ Slot 0 Value: 01
         └─ Memory Ring Grid Array: ['01', '00', '00', '00', '00', '00', '00', '00']

Cycle 01 | Hardware Pin Output: [10]
         ├─ Tracker: 📋 READ Slot 1 Value: 10
         └─ Memory Ring Grid Array: ['01', '10', '00', '00', '00', '00', '00', '00']

Cycle 02 | Hardware Pin Output: [01]
         ├─ Tracker: 📋 READ Slot 2 Value: 01
         └─ Memory Ring Grid Array: ['01', '10', '01', '00', '00', '00', '00', '00']

Cycle 03 | Hardware Pin Output: [Hi-Z]
         ├─ Tracker: ⚡ [GLOBAL RESET_N ACTIVE] All slots flushed to '00' (Neutral Potentials)
         └─ Memory Ring Grid Array: ['00', '00', '00', '00', '00', '00', '00', '00']

Cycle 04 | Hardware Pin Output: [01]
         ├─ Tracker: 📋 READ Slot 0 Value: 01
         └─ Memory Ring Grid Array: ['01', '00', '00', '00', '00', '00', '00', '00']

Cycle 05 | Hardware Pin Output: [10]
         ├─ Tracker